# Train SmolVLM-500M-Instruct



In [ ]:
!pip -q install "transformers>=4.47.0" peft bitsandbytes accelerate datasets pillow pandas numpy matplotlib seaborn scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/DL_final/data.zip" /content/data.zip
!unzip -q /content/data.zip -d /content/

In [ ]:
import os
import json
import math
import time
import random
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import re
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

sns.set_style("whitegrid")

In [ ]:
# ===== Config =====
SEED = 42
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

DATA_DIR = Path("/content")
OUT_DIR = Path("/content/drive/MyDrive/DL_final/artifacts/dora_layers_full")
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_SUBSET = False
SUBSET_FRAC = 0.2

RUN_NAME = "smolvlm_qdora_mc"
MAX_TRAINABLE_PARAMS = 5_000_000

EPOCHS = 10
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
LR = 1.0e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_TEXT_LENGTH = 1536
NUM_WORKERS = 2
LOG_EVERY = 50

HINT_DROPOUT_P = 0.2
LECTURE_DROPOUT_P = 0.2

DORA_R_CANDIDATES = [8]
DORA_ALPHA_MULT = 2
DORA_DROPOUT = 0.0

# Set True to run a small context sweep
ENABLE_SIMPLE_SWEEP = False
SWEEP_CONFIGS = [
    {"context_mode": "q_choices", "lr": 1.0e-4},
    {"context_mode": "q_hint", "lr": 1.0e-4},
    {"context_mode": "q_hint_lecture", "lr": 1.0e-4},
]

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

In [ ]:
# ===== Load data =====
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")

for df in [train_df, val_df]:
    df["choices"] = df["choices"].apply(json.loads)

if USE_SUBSET:
    train_df = train_df.sample(frac=SUBSET_FRAC, random_state=SEED).reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df))
print("Train num_choices:", train_df["num_choices"].value_counts().sort_index().to_dict())
print("Val num_choices:", val_df["num_choices"].value_counts().sort_index().to_dict())
print("Train answer dist:", train_df["answer"].value_counts().sort_index().to_dict())

In [ ]:
CHOICE_LETTERS = "ABCDEFG"

def resolve_image_path(data_dir: Path, rel_path: str) -> Path:
    p1 = data_dir / rel_path
    if p1.exists():
        return p1
    p2 = data_dir / "images" / rel_path
    if p2.exists():
        return p2
    p3 = data_dir / "images" / "images" / rel_path.replace("images/", "", 1)
    if p3.exists():
        return p3
    raise FileNotFoundError(f"Could not resolve image path: {rel_path}")

def build_user_text(row: pd.Series, choices: List[str], context_mode: str = "q_hint_lecture") -> str:
    context_parts = []
    hint = row.get("hint", "")
    lecture = row.get("lecture", "")

    if context_mode in ["q_hint", "q_hint_lecture"]:
        if pd.notna(hint) and str(hint).strip():
            context_parts.append(f"Hint: {str(hint).strip()}")

    if context_mode == "q_hint_lecture":
        if pd.notna(lecture) and str(lecture).strip():
            context_parts.append(f"Lecture: {str(lecture).strip()}")

    choice_lines = [f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)]

    parts = [
        "Answer the science multiple-choice question using the image.",
        "Respond with exactly one capital letter.",
    ]
    if context_parts:
        parts.append("\n".join(context_parts))
    parts.append(f"Question: {row['question']}")
    parts.append("Choices:\n" + "\n".join(choice_lines))

    return "\n\n".join(parts)

def build_messages(row: pd.Series, choices: List[str], context_mode: str = "q_hint_lecture"):
    text = build_user_text(row, choices, context_mode=context_mode)
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": text},
            ],
        }
    ]

In [ ]:
class ScienceQATrainDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, context_mode: str, hint_dropout_p: float, lecture_dropout_p: float, shuffle_choices: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.context_mode = context_mode
        self.hint_dropout_p = hint_dropout_p
        self.lecture_dropout_p = lecture_dropout_p
        self.shuffle_choices = shuffle_choices

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].copy()
        image = Image.open(resolve_image_path(self.data_dir, row["image_path"])).convert("RGB")

        if self.context_mode in ["q_hint", "q_hint_lecture"] and random.random() < self.hint_dropout_p:
            row["hint"] = ""
        if self.context_mode == "q_hint_lecture" and random.random() < self.lecture_dropout_p:
            row["lecture"] = ""

        choices = list(row["choices"])
        answer_idx = int(row["answer"])

        if self.shuffle_choices:
            perm = np.random.permutation(len(choices))
            shuffled = [choices[i] for i in perm]
            answer_idx = int(np.where(perm == answer_idx)[0][0])
            choices = shuffled

        messages = build_messages(row, choices, context_mode=self.context_mode)
        return {
            "image": image,
            "messages": messages,
            "answer_idx": answer_idx,
            "num_choices": len(choices),
        }

class ScienceQAValDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, context_mode: str):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.context_mode = context_mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(resolve_image_path(self.data_dir, row["image_path"])).convert("RGB")
        choices = list(row["choices"])
        messages = build_messages(row, choices, context_mode=self.context_mode)
        return {
            "id": row["id"],
            "image": image,
            "messages": messages,
            "answer_idx": int(row["answer"]),
            "num_choices": len(choices),
        }

In [ ]:
def load_processor_and_model(model_id: str):
    processor = AutoProcessor.from_pretrained(model_id)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )

    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )

    if not torch.cuda.is_available():
        model.to(device)

    model = prepare_model_for_kbit_training(model)
    return processor, model

In [ ]:
def find_lora_targets(model) -> List[str]:
    target_keywords = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    found = set()
    for name, _ in model.named_modules():
        for k in target_keywords:
            if name.endswith(k):
                found.add(k)
    return sorted(found)

def count_trainable_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def attach_dora_under_cap(model_loader_fn, max_trainable):
    targets = find_lora_targets(model_loader_fn()[1])
    if not targets:
        raise RuntimeError("No DoRA/LoRA targets found")

    for r in DORA_R_CANDIDATES:
        processor, fresh_model = model_loader_fn()

        dora_cfg = LoraConfig(
            r=r,
            lora_alpha=DORA_ALPHA_MULT * r,
            lora_dropout=DORA_DROPOUT,
            bias="none",
            target_modules=targets,
            rank_pattern={
                "q_proj": 8,
                "k_proj": 8,
                "v_proj": 8,
                "o_proj": 8,
                "gate_proj": 7,
                "up_proj": 7,
                "down_proj": 7,
            },
            alpha_pattern={
                "q_proj": 16,
                "k_proj": 16,
                "v_proj": 16,
                "o_proj": 16,
                "gate_proj": 14,
                "up_proj": 14,
                "down_proj": 14,
            },
            task_type="CAUSAL_LM",
            use_dora=True,
        )

        peft_model = get_peft_model(fresh_model, dora_cfg)
        tp = count_trainable_params(peft_model)

        print(f"DoRA rank={r} | targets={targets} | trainable_params={tp:,}")
        peft_model.print_trainable_parameters()

        if tp <= max_trainable:
            return processor, peft_model, r, tp, targets

        del peft_model
        del fresh_model
        torch.cuda.empty_cache()

    raise RuntimeError("No DoRA rank satisfies the trainable parameter cap")

In [ ]:
def move_to_model_device(inputs: Dict[str, torch.Tensor], model):
    model_device = next(model.parameters()).device
    out = {}
    for k, v in inputs.items():
        out[k] = v.to(model_device) if torch.is_tensor(v) else v
    return out

def make_letter_token_id_map(tokenizer):
    token_map = {}
    for ch in CHOICE_LETTERS:
        token_ids = []
        for variant in [ch, " " + ch]:
            ids = tokenizer(variant, add_special_tokens=False).input_ids
            if len(ids) == 1:
                token_ids.append(ids[0])
        token_ids = sorted(set(token_ids))
        if not token_ids:
            raise ValueError(f"No single-token form found for {ch}")
        token_map[ch] = token_ids
    return token_map

def collate_fn(batch, processor):
    images = [x["image"] for x in batch]
    texts = [processor.apply_chat_template(x["messages"], add_generation_prompt=True) for x in batch]
    answer_idx = torch.tensor([x["answer_idx"] for x in batch], dtype=torch.long)
    num_choices = torch.tensor([x["num_choices"] for x in batch], dtype=torch.long)

    proc = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )
    return {
        "inputs": proc,
        "answer_idx": answer_idx,
        "num_choices": num_choices,
    }

@torch.no_grad()
def evaluate_mc(model, processor, val_loader, letter_token_map):
    model.eval()
    model_device = next(model.parameters()).device

    total = 0
    correct = 0
    nll_sum = 0.0
    letter_ce_sum = 0.0

    val_bar = tqdm(val_loader, total=len(val_loader), desc="Validation", leave=False)

    for batch in val_bar:
        inputs = move_to_model_device(batch["inputs"], model)
        answer_idx = batch["answer_idx"].to(model_device)
        num_choices = batch["num_choices"].to(model_device)

        out = model(**inputs)
        logits = out.logits
        attn = inputs["attention_mask"]
        last_idx = attn.sum(dim=1) - 1
        next_logits = logits[torch.arange(logits.size(0), device=model_device), last_idx, :]

        log_probs = F.log_softmax(next_logits, dim=-1)

        max_choices = len(CHOICE_LETTERS)
        choice_log_probs = torch.full((logits.size(0), max_choices), -1e9, device=model_device)
        for j, ch in enumerate(CHOICE_LETTERS):
            tok_ids = torch.tensor(letter_token_map[ch], device=model_device)
            choice_log_probs[:, j] = torch.logsumexp(log_probs[:, tok_ids], dim=1)

        invalid_mask = torch.arange(max_choices, device=model_device).unsqueeze(0) >= num_choices.unsqueeze(1)
        masked = choice_log_probs.masked_fill(invalid_mask, -1e9)

        preds = masked.argmax(dim=1)
        correct += (preds == answer_idx).sum().item()
        total += logits.size(0)

        nll_sum += F.cross_entropy(masked, answer_idx, reduction="sum").item()

        target_scores = []
        for i in range(log_probs.size(0)):
            ch = CHOICE_LETTERS[int(answer_idx[i].item())]
            tok_ids = torch.tensor(letter_token_map[ch], device=model_device)
            target_scores.append(torch.logsumexp(log_probs[i, tok_ids], dim=0))
        target_scores = torch.stack(target_scores)
        letter_ce_sum += (-target_scores).sum().item()

    return {
        "val_mc_accuracy": correct / max(total, 1),
        "val_mc_nll": nll_sum / max(total, 1),
        "val_loss_letter": letter_ce_sum / max(total, 1),
    }

In [ ]:
def train_one_run(run_name: str, context_mode: str, lr: float):
    print(f"\n=== RUN {run_name} | context_mode={context_mode} | lr={lr} ===")

    processor, model, r_used, tp_used, targets_used = attach_dora_under_cap(
    lambda: load_processor_and_model(MODEL_ID),
    MAX_TRAINABLE_PARAMS,
)

    train_ds = ScienceQATrainDataset(
        train_df, DATA_DIR, context_mode=context_mode,
        hint_dropout_p=HINT_DROPOUT_P, lecture_dropout_p=LECTURE_DROPOUT_P, shuffle_choices=True,
    )
    val_ds = ScienceQAValDataset(val_df, DATA_DIR, context_mode=context_mode)

    train_loader = DataLoader(
        train_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=lambda b: collate_fn(b, processor),
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=lambda b: collate_fn(b, processor),
        pin_memory=torch.cuda.is_available(),
    )

    letter_token_map = make_letter_token_id_map(processor.tokenizer)
    model_device = next(model.parameters()).device

    optim_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(optim_params, lr=lr, weight_decay=WEIGHT_DECAY)

    total_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    history = []

    global_step = 0
    start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        optimizer.zero_grad(set_to_none=True)

        progress_bar = tqdm(
            enumerate(train_loader, start=1),
            total=len(train_loader),
            desc=f"{run_name} | Epoch {epoch}/{EPOCHS}",
            leave=True,
        )

        for step, batch in progress_bar:
            inputs = move_to_model_device(batch["inputs"], model)
            answer_idx = batch["answer_idx"].to(model_device)
            num_choices = batch["num_choices"].to(model_device)

            out = model(**inputs)
            logits = out.logits
            attn = inputs["attention_mask"]
            last_idx = attn.sum(dim=1) - 1
            next_logits = logits[torch.arange(logits.size(0), device=model_device), last_idx, :]

            log_probs = F.log_softmax(next_logits, dim=-1)
            choice_scores = torch.full((logits.size(0), len(CHOICE_LETTERS)), -1e9, device=model_device)
            for j, ch in enumerate(CHOICE_LETTERS):
                tok_ids = torch.tensor(letter_token_map[ch], device=model_device)
                choice_scores[:, j] = torch.logsumexp(log_probs[:, tok_ids], dim=1)

            invalid_mask = torch.arange(len(CHOICE_LETTERS), device=model_device).unsqueeze(0) >= num_choices.unsqueeze(1)
            choice_scores = choice_scores.masked_fill(invalid_mask, -1e9)

            loss = F.cross_entropy(choice_scores, answer_idx)
            (loss / GRAD_ACCUM_STEPS).backward()

            running_loss += loss.item() * logits.size(0)
            seen += logits.size(0)

            progress_bar.set_postfix({
                "loss": f"{running_loss / max(seen, 1):.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            })

            if step % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1


        if len(train_loader) % GRAD_ACCUM_STEPS != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate_mc(model, processor, val_loader, letter_token_map)

        row = {
            "run_name": run_name,
            "epoch": epoch,
            "global_step": global_step,
            "train_loss": train_loss,
            "val_loss_letter": val_metrics["val_loss_letter"],
            "val_mc_accuracy": val_metrics["val_mc_accuracy"],
            "val_mc_nll": val_metrics["val_mc_nll"],
            "lr": scheduler.get_last_lr()[0],
            "trainable_params": tp_used,
            "dora_r": r_used,
            "context_mode": context_mode,
        }
        history.append(row)

        print(
            f"[Epoch {epoch}] train={train_loss:.4f} | "
            f"val_letter={val_metrics['val_loss_letter']:.4f} | "
            f"val_acc={val_metrics['val_mc_accuracy']:.4f} | "
            f"val_nll={val_metrics['val_mc_nll']:.4f}"
        )

        checkpoint_dir = OUT_DIR / run_name / f"checkpoint_epoch_{epoch:02d}"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        model.save_pretrained(checkpoint_dir)
        processor.save_pretrained(checkpoint_dir)

        with open(checkpoint_dir / "run_config.json", "w") as f:
            json.dump(
                {
                    "run_name": run_name,
                    "model_id": MODEL_ID,
                    "epoch": int(epoch),
                    "global_step": int(global_step),
                    "lora_r": int(r_used),   # or "dora_r" if you renamed it
                    "trainable_params": int(tp_used),
                    "context_mode": context_mode,
                    "lr": lr,
                    "max_text_length": MAX_TEXT_LENGTH,
                    "val_loss_letter": float(val_metrics["val_loss_letter"]),
                    "val_mc_accuracy": float(val_metrics["val_mc_accuracy"]),
                    "val_mc_nll": float(val_metrics["val_mc_nll"]),
                },
                f,
                indent=2,
            )

        print(f"Saved checkpoint: {checkpoint_dir}")

    hist_df = pd.DataFrame(history)
    best_row = hist_df.sort_values(
        ["val_mc_accuracy", "val_mc_nll"],
        ascending=[False, True],
    ).iloc[0]

    best_epoch = int(best_row["epoch"])
    best_val_acc = float(best_row["val_mc_accuracy"])
    best_val_nll = float(best_row["val_mc_nll"])

    summary = {
        "run_name": run_name,
        "best_val_mc_accuracy": best_val_acc,
        "best_val_mc_nll": best_val_nll,
        "best_epoch": best_epoch,
        "best_checkpoint_dir": str(OUT_DIR / run_name / f"checkpoint_epoch_{best_epoch:02d}"),
        "lora_r": int(r_used),  # or "dora_r" if using DoRA naming
        "trainable_params": int(tp_used),
        "context_mode": context_mode,
        "lr": lr,
        "runtime_minutes": (time.time() - start) / 60.0,
    }
    return hist_df, pd.DataFrame([summary])

In [ ]:
# ===== Sanity check prompt rendering and tokenization =====
tmp_processor = AutoProcessor.from_pretrained(MODEL_ID)
sample_row = train_df.iloc[0]
sample_messages = build_messages(sample_row, list(sample_row["choices"]), context_mode="q_hint_lecture")
rendered = tmp_processor.apply_chat_template(sample_messages, add_generation_prompt=True)
print(rendered[:3000])

for ch in CHOICE_LETTERS[:len(sample_row["choices"])]:
    raw_ids = tmp_processor.tokenizer(ch, add_special_tokens=False).input_ids
    spaced_ids = tmp_processor.tokenizer(" " + ch, add_special_tokens=False).input_ids
    print(f"{ch}: raw={raw_ids}, spaced={spaced_ids}")

In [ ]:
all_hist = []
all_sum = []

if ENABLE_SIMPLE_SWEEP:
    run_cfgs = SWEEP_CONFIGS
else:
    run_cfgs = [{"context_mode": "q_hint_lecture", "lr": LR}]

for i, cfg in enumerate(run_cfgs, start=1):
    run_name = f"{RUN_NAME}_run{i}_{cfg['context_mode']}_lr{cfg['lr']}"
    h, s = train_one_run(run_name, cfg["context_mode"], cfg["lr"])
    all_hist.append(h)
    all_sum.append(s)

metrics_history = pd.concat(all_hist, ignore_index=True)
experiment_summary = pd.concat(all_sum, ignore_index=True)

metrics_path = OUT_DIR / "metrics_history.csv"
summary_path = OUT_DIR / "experiment_summary.csv"
metrics_history.to_csv(metrics_path, index=False)
experiment_summary.to_csv(summary_path, index=False)

print("Saved:", metrics_path)
print("Saved:", summary_path)
display(experiment_summary.sort_values(["best_val_mc_nll", "best_val_mc_accuracy"], ascending=[True, False]))

In [ ]:
best_row = experiment_summary.sort_values(
    ["best_val_mc_accuracy", "best_val_mc_nll"],
    ascending=[False, True],
).iloc[0]

BEST_RUN_NAME = best_row["run_name"]
BEST_ADAPTER_DIR = Path(best_row["best_checkpoint_dir"])

print("Best run:", BEST_RUN_NAME)
print("Best checkpoint dir:", BEST_ADAPTER_DIR)
print(best_row.to_dict())

In [ ]:
# ===== Evidence plots =====
mh = metrics_history.copy()
es = experiment_summary.copy()

# 1) Train/Val loss curves
plt.figure(figsize=(8, 5))
for rn, g in mh.groupby("run_name"):
    plt.plot(g["epoch"], g["train_loss"], marker="o", label=f"{rn} train")
    plt.plot(g["epoch"], g["val_loss_letter"], marker="x", linestyle="--", label=f"{rn} val")
plt.title("Train vs Validation Letter Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(fontsize=8)
plt.show()

# 2) Val accuracy
plt.figure(figsize=(8, 5))
for rn, g in mh.groupby("run_name"):
    plt.plot(g["epoch"], g["val_mc_accuracy"], marker="o", label=rn)
plt.title("Validation MC Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(fontsize=8)
plt.show()

# 3) Val MC NLL
plt.figure(figsize=(8, 5))
for rn, g in mh.groupby("run_name"):
    plt.plot(g["epoch"], g["val_mc_nll"], marker="o", label=rn)
plt.title("Validation MC NLL (lower is better)")
plt.xlabel("Epoch")
plt.ylabel("NLL")
plt.legend(fontsize=8)
plt.show()

# 4) Best metrics by run
fig, ax1 = plt.subplots(figsize=(9, 5))
x = np.arange(len(es))
ax1.bar(x - 0.2, es["best_val_mc_accuracy"], width=0.4, label="Best Val Acc")
ax2 = ax1.twinx()
ax2.bar(x + 0.2, es["best_val_mc_nll"], width=0.4, color="orange", label="Best Val NLL")
ax1.set_xticks(x)
ax1.set_xticklabels(es["run_name"], rotation=30, ha="right")
ax1.set_ylabel("Accuracy")
ax2.set_ylabel("NLL")
ax1.set_title("Best Validation Metrics by Run")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()

# 5) Accuracy vs trainable params
plt.figure(figsize=(7, 5))
plt.scatter(es["trainable_params"], es["best_val_mc_accuracy"], s=80)
for _, r in es.iterrows():
    plt.text(r["trainable_params"], r["best_val_mc_accuracy"], r["run_name"], fontsize=8)
plt.axvline(MAX_TRAINABLE_PARAMS, color="red", linestyle="--", label="5M cap")
plt.title("Best Val Accuracy vs Trainable Params")
plt.xlabel("Trainable Params")
plt.ylabel("Best Val Accuracy")
plt.legend()
plt.show()

In [ ]:
with open(OUT_DIR / "best_checkpoint.txt", "w") as f:
    f.write(str(BEST_ADAPTER_DIR) + "\n")

print("Saved best checkpoint marker:", OUT_DIR / "best_checkpoint.txt")
print("Best checkpoint:", BEST_ADAPTER_DIR)